# 02 - Pré-processamento de Texto

**PA007 · NLP Análise de Sentimento Zomato**

**Fase CRISP-DM:** Data Preparation (Clean/Construct Data)

**Input:** `data/processed/zomato_reviews_clean.csv` (3.370 reviews, saída do nb01)
**Output:** `data/processed/zomato_reviews_processed.csv` (+ coluna `review_processed`)

**Pipeline:** `review` → `clean_text` → `tokenize` → `remove_stopwords` → `lemmatize` → `review_processed`, funções reaproveitadas de `src/preprocessing.py`.


## 1. Setup e imports

In [81]:
import sys
sys.path.insert(0, "..")

import time
import csv
import pandas as pd
import nltk
from collections import Counter
from itertools import chain
from scipy.stats import kruskal

for resource in ["stopwords", "wordnet", "omw-1.4"]:
    nltk.download(resource, quiet=True)

from src.preprocessing import (
    clean_text, tokenize, remove_stopwords, lemmatize, preprocess_pipeline
)

print("Setup concluído.")


Setup concluído.


## 2. Carregamento dos dados

In [82]:
df = pd.read_csv("../data/processed/zomato_reviews_clean.csv")
print(f"Shape: {df.shape}")
print(f"Colunas: {df.columns.tolist()}")
print()
print(df["sentiment"].value_counts())
df.head(3)


Shape: (3370, 3)
Colunas: ['rating', 'review', 'sentiment']

sentiment
positivo    1642
negativo    1439
neutro       289
Name: count, dtype: int64


,rating,review,sentiment
0,5,"best biryani , so supportive staff of outlet ,...",positivo
1,4,delivery boy was very decent and supportive.👌👍,positivo
2,1,"worst biryani i have tasted in my life, half o...",negativo


## 3. Pipeline de limpeza, passo a passo

Demonstração de cada etapa numa review real com tag `<br/>`, o mesmo problema que motivou a correção do `tokenizar()` local no nb01.


In [83]:
mask_br = df["review"].str.contains(r"<br\s*/?>", regex=True, case=False, na=False)
example_idx = df[mask_br].index[0]
original = df.loc[example_idx, "review"]

step1 = clean_text(original)
step2 = tokenize(step1)
step3 = remove_stopwords(step2)
step4 = lemmatize(step3)

print(f"[0] Original:       {original}")
print(f"[1] clean_text:     {step1}")
print(f"[2] tokenize:       {step2}")
print(f"[3] sem stopwords:  {step3}")
print(f"[4] lematizado:     {step4}")
print()
print(f"Resumo: {len(step2)} tokens -> {len(step4)} após filtragem")


[0] Original:       Pizza hut vari bad bro😡😡😡😡😡<br/>All pablik pizzz not send pizz hat oder <br/>Vari vari bad test
[1] clean_text:     pizza hut vari bad bro all pablik pizzz not send pizz hat oder vari vari bad test
[2] tokenize:       ['pizza', 'hut', 'vari', 'bad', 'bro', 'all', 'pablik', 'pizzz', 'not', 'send', 'pizz', 'hat', 'oder', 'vari', 'vari', 'bad', 'test']
[3] sem stopwords:  ['pizza', 'hut', 'vari', 'bad', 'bro', 'pablik', 'pizzz', 'send', 'pizz', 'hat', 'oder', 'vari', 'vari', 'bad', 'test']
[4] lematizado:     ['pizza', 'hut', 'vari', 'bad', 'bro', 'pablik', 'pizzz', 'send', 'pizz', 'hat', 'oder', 'vari', 'vari', 'bad', 'test']

Resumo: 17 tokens -> 15 após filtragem


`clean_text` substitui a tag `<br/>` por espaço antes de remover pontuação, as palavras vizinhas ficam separadas corretamente.

## 4. Aplicação ao dataset completo

In [84]:
start = time.time()
df["tokens"] = df["review"].apply(preprocess_pipeline)
df["review_processed"] = df["tokens"].apply(" ".join)
elapsed = time.time() - start

print(f"Pipeline aplicada em {elapsed:.2f}s | Shape: {df.shape}")
df[["review", "review_processed"]].head(5)


Pipeline aplicada em 0.13s | Shape: (3370, 5)


,review,review_processed
0,"best biryani , so supportive staff of outlet ,...",best biryani supportive staff outlet personali...
1,delivery boy was very decent and supportive.👌👍,delivery boy decent supportive
2,"worst biryani i have tasted in my life, half o...",worst biryani tasted life half biryani dustbin
3,all food is good and tasty . will order again ...,food good tasty order lot explore bawarchi menu
4,shandar zabardast zindabad .. good going bawar...,shandar zabardast zindabad good going bawarchi...


## 5. Diagnóstico pós-processamento

In [85]:
df["n_words_orig"] = df["review"].str.split().str.len()
df["n_tokens_proc"] = df["tokens"].str.len()

stats = (
    df.groupby("sentiment")[["n_words_orig", "n_tokens_proc"]]
    .agg(["mean", "median"])
    .round(1)
)
print("Tokens por classe, antes e depois do pré-processamento:")
print(stats)


Tokens por classe, antes e depois do pré-processamento:
          n_words_orig        n_tokens_proc       
                  mean median          mean median
sentiment                                         
negativo          12.5    9.0           7.5    5.0
neutro            12.3    9.0           7.4    5.0
positivo          13.4   10.0           7.9    6.0


In [86]:
all_tokens = list(chain.from_iterable(df["tokens"]))
vocab = set(all_tokens)

orig_mean = df["n_words_orig"].mean()
proc_mean = df["n_tokens_proc"].mean()
reduction_pct = (1 - proc_mean / orig_mean) * 100

print(f"Total de tokens (com repetição): {len(all_tokens):,}")
print(f"Vocabulário único:               {len(vocab):,}")
print(f"Redução média por review:        {orig_mean:.1f} -> {proc_mean:.1f} tokens ({reduction_pct:.1f}% menos)")


Total de tokens (com repetição): 25,965
Vocabulário único:               4,188
Redução média por review:        12.9 -> 7.7 tokens (40.2% menos)


In [87]:
for label in ["positivo", "neutro", "negativo"]:
    subset = list(chain.from_iterable(df[df["sentiment"] == label]["tokens"]))
    top20 = Counter(subset).most_common(20)
    tokens_str = "  ".join([f"{w}({c})" for w, c in top20])
    print(f"\n{label.upper()}:")
    print(tokens_str)



POSITIVO:
food(367)  good(345)  taste(311)  order(216)  bad(153)  quality(134)  delivery(120)  quantity(111)  time(102)  service(100)  like(96)  ordered(95)  restaurant(93)  le(88)  also(77)  worst(76)  test(73)  money(70)  best(68)  experience(66)

NEUTRO:
food(60)  taste(57)  good(54)  bad(45)  order(33)  delivery(29)  money(28)  restaurant(24)  quality(22)  best(22)  service(19)  also(17)  worst(16)  quantity(15)  time(14)  nice(13)  please(13)  tha(13)  le(13)  late(12)

NEGATIVO:
food(277)  good(270)  taste(237)  order(197)  bad(135)  quality(112)  delivery(105)  like(99)  quantity(96)  test(94)  restaurant(92)  service(88)  time(82)  ordered(79)  worst(76)  money(76)  chicken(68)  best(66)  le(63)  also(63)


### **5.1 (H1) Redução de vocabulário afeta as três classes de forma equilibrada**

Se a limpeza (stopwords + lematização) afetasse as três classes de forma equilibrada, a taxa de redução de tokens devia ser parecida entre positivo, neutro e negativo. O teste pergunta: alguma classe perde proporcionalmente mais sinal textual que as outras, ou a diferença observada cabe na variação natural da amostra?

Teste: Kruskal-Wallis comparando a taxa de redução (`1 - n_tokens_proc / n_words_orig`) entre as três classes.


In [88]:
df["taxa_reducao"] = 1 - (df["n_tokens_proc"] / df["n_words_orig"])

grupos = [g["taxa_reducao"].values for _, g in df.groupby("sentiment")]
h_stat, p_value = kruskal(*grupos)

print("Taxa de redução média por classe:")
print(df.groupby("sentiment")["taxa_reducao"].mean().round(3))
print()
print(f"H = {h_stat:.2f} | p-value = {p_value:.4f}")


Taxa de redução média por classe:
sentiment
negativo    0.364
neutro      0.359
positivo    0.375
Name: taxa_reducao, dtype: float64

H = 4.52 | p-value = 0.1043


**Resultado:** não, a diferença não é grande demais pra ser acaso (H=4,52; p=0,104). As três classes perdem proporção parecida de tokens (negativo 36,4%, neutro 35,9%, positivo 37,5%). A limpeza não distorce nenhuma classe desproporcionalmente, o vocabulário reduzido continua comparável entre positivo, neutro e negativo.

**H1 não rejeitada.** A padronização do texto (stopwords + lematização) é um tratamento neutro em relação à classe, não introduz viés de limpeza que favoreça ou prejudique alguma categoria de sentimento.


### 5.2 Investigação de tokens suspeitos: `le`, `test`, `tha`, `amp`(detectado antes de ajustar no .py)

Três tokens de alta frequência chamam atenção só pela forma, sem supor o que significam: `le` (2 caracteres, top-20 nas três classes), `test` (parece a palavra inglesa comum, mas o domínio é comida) e `tha` (não é palavra inglesa nem stopword NLTK). Cada um vira uma pergunta: é ruído do pipeline de limpeza, ou carrega sinal legítimo escrito de um jeito diferente do esperado?

**(H2) - `le` é artefato determinístico da lematização de "less", não ruído.**

Se `le` fosse ruído aleatório (erro de digitação, sigla, fragmento sem relação com inglês), o texto original das reviews que geram esse token não teria relação sistemática com nenhuma palavra reconhecível. Se for artefato do pipeline, o texto original deveria conter a palavra "less" em praticamente 100% dos casos, e o mecanismo deveria ser reproduzível de forma determinística.

In [89]:
from nltk.corpus import stopwords as nltk_sw
from nltk.stem import WordNetLemmatizer

lem = WordNetLemmatizer()
sw_en = set(nltk_sw.words("english"))

le_mask = df["tokens"].apply(lambda toks: "le" in toks)
n_le = le_mask.sum()

print(f"Reviews com token 'le': {n_le}")
print()
print("Amostra de reviews com 'le':")
print(df[le_mask][["review", "sentiment"]].head(8).to_string())

Reviews com token 'le': 158

Amostra de reviews com 'le':
                                                                                                                       review sentiment
19                                                                             Taste less nd took 55 min to deliver the order  negativo
41                                                                              quality not much good and quantity  are less   negativo
50                                   HAD GIVEN VERY VERY VERY VERY LESS CHILLI FLAKES AND NO OREGANO AT ALL VERY DISSAPOINTED  positivo
77                  very bad taste taste less saltless ,they don't even accept food instructions, no cutlery and very costly   positivo
125                                                                                           size of both item is very less   negativo
144                                            sent less number of breads this time. I will never give order again to Gangor  

In [90]:
contains_less = df.loc[le_mask, "review"].str.contains(r"\bless\b", case=False, regex=True)
pct_less = contains_less.mean() * 100

print(f"Dessas, contêm 'less' no texto original: {contains_less.sum()} ({pct_less:.1f}%)")

print("--- Mecanismo ---")
print(f"'less' in stopwords NLTK: {'less' in sw_en}  -> passa pelo filtro de stopwords")
print(f"lemmatize('less', pos='n') [padrão]: {lem.lemmatize('less', pos='n')!r}")
print(f"lemmatize('less', pos='r') [advérbio, correto]: {lem.lemmatize('less', pos='r')!r}")
print()
if contains_less.sum() < n_le:
    print("Reviews com 'le' que NÃO contêm 'less' no texto original:")
    print(df.loc[le_mask][~contains_less][["review", "sentiment"]].to_string())

Dessas, contêm 'less' no texto original: 153 (96.8%)
--- Mecanismo ---
'less' in stopwords NLTK: False  -> passa pelo filtro de stopwords
lemmatize('less', pos='n') [padrão]: 'le'
lemmatize('less', pos='r') [advérbio, correto]: 'less'

Reviews com 'le' que NÃO contêm 'less' no texto original:
                                                                                                                                                                 review sentiment
235                                                                                                                plz dhayan de pese le rahe ho to wesa bana ke bhi do  positivo
547                   Worst food test lesss wala paisa Waisted he bhi koi magana matt Yaha se ghatiya hain khud test kr ke bol rha hun bohot hi jyada worsted test hain  negativo
619   So tasty in malhar dosa it's food are so spicy and very good and serves so aap cool and are special le dosa has the very tasty and mysore dosa has the so yummies  pos

**Resultado:** sim, é artefato determinístico. Das 158 reviews com o token `le`, 153 (96,8%) contêm a palavra "less" no texto original. As 5 exceções (3,2%) são casos correlatos, grafias como "lesss" e "lesses" (erro de digitação de "less", não capturado pelo `\bless\b`) e um caso de "le" como palavra hindi solta ("carj le rahe he"), coincidência de forma, não do mesmo mecanismo.

**H2 confirmada.** `le` não é ruído nem sinal próprio, é a saída de `WordNetLemmatizer().lemmatize('less', pos='n')`, que interpreta "less" como substantivo plural e devolve uma forma inventada. Como "less" não está em `stopwords.words('english')`, o token chega inteiro ao lematizador. Na leitura de feature importance do nb03, `le` deve ser lido como "less".

**(H3) - `test` é grafia fonética Hinglish de "taste", não a palavra inglesa comum.**

Se `test` fosse a palavra inglesa comum (exame, ensaio, teste de qualidade), eu esperaria conviver normalmente com o token "taste" na mesma review, já que seriam ideias diferentes. Se for grafia alternativa de "taste", eu esperaria baixa coocorrência com "taste" na mesma review, porque normalmente é a mesma ideia escrita de um jeito só, e contexto claramente ligado a sabor de comida nas amostras.

In [91]:
test_mask = df["tokens"].apply(lambda toks: "test" in toks)
n_test = test_mask.sum()

print(f"Reviews com token 'test': {n_test}")
print()
print("Amostra de reviews com 'test':")
print(df[test_mask][["review", "sentiment"]].sample(8, random_state=42).to_string())

Reviews com token 'test': 160

Amostra de reviews com 'test':
                                                                                                                   review sentiment
1920                                                                                                 very bad kharab test  negativo
1987                                                                                    test good and quantity it's avg.   positivo
2947                                                                                                test great but mrp208  negativo
888                                                                                         less quantity and poor test..  negativo
1738                                                                                            all dhokla tests are same  positivo
469   sabji alag bheja he or aachar (pickle) to aaya hi nahi he test everage packing are without beg only dish hand over   negativo
1849          

In [92]:
taste_mask = df["tokens"].apply(lambda toks: "taste" in toks)
both = (test_mask & taste_mask).sum()
only_test = (test_mask & ~taste_mask).sum()

print()
print(f"Também têm 'taste' na mesma review: {both} ({both/n_test*100:.1f}%)")
print(f"Só têm 'test' (sem 'taste'):         {only_test} ({only_test/n_test*100:.1f}%)")


Também têm 'taste' na mesma review: 1 (0.6%)
Só têm 'test' (sem 'taste'):         159 (99.4%)


**Resultado:** sim. Das 160 reviews com o token `test`, apenas 1 (0,6%) também contém "taste" na mesma review, quase exclusão mútua, o oposto do que se esperaria se fossem duas palavras diferentes usadas normalmente lado a lado. As amostras confirmam o padrão de escrita fonética indiana para "taste" (ex: *"kharab test"*, *"test good and quantity"*, *"such bad test"*), sempre em contexto de sabor de comida, nunca no sentido de exame/ensaio.

**H3 confirmada.** `test` é grafia Hinglish alternativa de "taste". Token mantido como está, o TF-IDF do nb03 vai tratá-lo como feature normal, mas a leitura de "test" nas features deve considerar essa equivalência.

**(H4) - `tha` é palavra hindi transliterada (था = "was"), Hinglish legítimo, não fragmento de tokenização.**

Se `tha` fosse fragmento de erro de tokenização (pedaço cortado de outra palavra, ruído de caracteres), o padrão de aparição seria disperso e sem relação com nenhuma palavra reconhecível, e apareceria colado ou dentro de outras palavras. Se for a palavra funcional hindi "tha" (auxiliar verbal, equivalente a "was"/"foi"), eu esperaria encontrá-la sempre como token isolado, em reviews com conteúdo claramente Hinglish (mistura de inglês com hindi transliterado).

In [93]:
tha_mask = df["tokens"].apply(lambda toks: "tha" in toks)
print(f"Reviews com token 'tha': {tha_mask.sum()}")
print(df.loc[tha_mask, "sentiment"].value_counts())
print()
print("Amostra de reviews com 'tha':")
print(df[tha_mask][["review", "sentiment"]].head(10).to_string())


Reviews com token 'tha': 88
sentiment
positivo    39
negativo    37
neutro      12
Name: count, dtype: int64

Amostra de reviews com 'tha':
                                                                                                               review sentiment
67                                                                                                    kitna oily tha   negativo
108                                                               🥄 spoon nhi tha yahi hai buss<br/>wrna or sab msttt  negativo
111                                      late dilivery and food boht bakvas tha <br/>never order from this restaurant  negativo
123                                                                                      6 bola tha fir bi only 1 pc   positivo
182                                                             Sandwich ka taste bhi accha nhi tha or 2-3 baal nikle    neutro
190  The food was totaly bad in  taste… fried rice to pura jal gya hua tha bekar tha… <br/>C

In [94]:
print("--- Outros tokens curtos (2-3 chars) de alta frequência, mesma família Hinglish? ---")
short_tokens = [(t, c) for t, c in Counter(all_tokens).most_common() if len(t) in (2, 3)]
for tok, cnt in short_tokens[:12]:
    print(f"  {tok!r}: {cnt}x")

--- Outros tokens curtos (2-3 chars) de alta frequência, mesma família Hinglish? ---
  'bad': 333x
  'le': 164x
  'tha': 112x
  'one': 84x
  'hai': 77x
  'eat': 72x
  'got': 70x
  'try': 69x
  'boy': 65x
  'bhi': 55x
  'hot': 54x
  'dal': 48x


**Resultado:** sim. 88 reviews têm o token `tha` (39 positivo, 37 negativo, 12 neutro), sempre isolado entre outras palavras e sempre em reviews com mistura clara de inglês e hindi transliterado (ex: *"kitna oily tha"*, *"food boht bakvas tha"*, *"bargar thanda ho chuka tha"*), nunca colado dentro de uma palavra maior. Junto de tokens curtos vizinhos (`hi` 36x, `nd` 34x, `ka` 31x, `ki` 24x, `ke` 22x, `ho` 18x), forma uma família de partículas funcionais hindi que passam pelo filtro por ele ser uma lista de stopwords só em inglês.

**H4 confirmada.** `tha` é palavra funcional hindi (था, auxiliar verbal "was"/"foi"), Hinglish legítimo, não fragmento de tokenização. Mesma decisão de manter Hinglish tomada no nb01 se aplica aqui, não é erro do pipeline nem exige stopwords customizado nesta fase, curar uma lista de stopwords em hindi está fora do escopo do baseline.

**Conclusão da investigação (H2-H4):** nenhum dos três tokens é ruído do pipeline. `le` é artefato determinístico e documentado da lematização de "less" (sinal preservado, interpretabilidade reduzida). `test` é grafia Hinglish alternativa de "taste", tratado como token normal pelo TF-IDF. `tha` é palavra funcional hindi, mantida pela decisão já tomada de não filtrar Hinglish. Nenhuma ação de stopwords customizado necessária nesta fase.

**(H5) - `amp` é resíduo de entidade HTML não decodificada, não sinal legítimo como `tha`.**

Na mesma investigação de tokens curtos que sustentou a H4 (célula acima, ranking de tokens de 2-3 chars), `amp` também apareceu com volume alto, logo ao lado de `tha`, antes da correção mostrada a seguir ser aplicada ao pipeline:

```
'bad': 333x  'le': 164x  'tha': 112x  'amp': 92x  'one': 84x  'hai': 77x  'eat': 72x  'got': 70x  ...
```

Mesma pergunta feita pras outras três: é sinal legítimo escrito de um jeito diferente do esperado, como `le`/`test`/`tha`, ou é ruído do pipeline de limpeza? Se fosse sinal, eu esperaria uma palavra reconhecível por trás dele. Se fosse ruído técnico, eu esperaria achar uma causa mecânica no texto bruto, um padrão que o pipeline processa errado, não uma palavra em inglês ou Hinglish de verdade.

In [100]:
import re

def old_clean_text(text):
    text = str(text).lower()
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

old_tokens = df["review"].apply(lambda t: old_clean_text(t).split())
amp_mask = old_tokens.apply(lambda toks: "amp" in toks)
n_amp = amp_mask.sum()

print(f"Reviews com token 'amp' pelo pipeline antigo (sem html.unescape): {n_amp}")
print("Amostra de reviews com 'amp':")
print(df[amp_mask][["review", "sentiment"]].head(8).to_string())

contains_amp_entity = df.loc[amp_mask, "review"].str.contains("&amp;", case=False, regex=False)
pct = contains_amp_entity.mean() * 100
print(f"Dessas, contêm a entidade '&amp;' no texto bruto: {contains_amp_entity.sum()} ({pct:.1f}%)")

Reviews com token 'amp' pelo pipeline antigo (sem html.unescape): 68
Amostra de reviews com 'amp':
                                                                                                                                                                                                                  review sentiment
133                                                                                                                                                                     pizza packing &amp; test is desi. coco excellent  negativo
159                                                                                                                                                                       tast is better &amp; service is very satisfied  positivo
255                                                                                                                                                                   delivery timing is good &amp; food is very swadist  po

In [98]:
example_raw = df.loc[amp_mask, "review"].iloc[0]
print("--- Mecanismo, antes x depois do fix, num exemplo real ---")
print(f"Original:                       {example_raw}")
print(f"clean_text SEM html.unescape(): {old_clean_text(example_raw)}")
print(f"clean_text ATUAL (com fix):     {clean_text(example_raw)}")

--- Mecanismo, antes x depois do fix, num exemplo real ---
Original:                       pizza packing &amp; test is desi. coco excellent
clean_text SEM html.unescape(): pizza packing amp test is desi coco excellent
clean_text ATUAL (com fix):     pizza packing test is desi coco excellent


**Resultado:** sim, é ruído técnico. Reconstruindo o pipeline antigo (sem `html.unescape()`), 68 reviews geram o token `amp`, e `&amp;` aparece em todas elas antes da limpeza. Confirmando: as 68 (100%) contêm a entidade `&amp;` no texto bruto, "&" escrito como código de página web. O regex de limpeza removia tags (`<...>`) mas não decodificava entidades como essa antes de tirar caracteres não alfabéticos, sobravam só as letras do meio, "amp", como se fosse uma palavra.

**H5 confirmada, na direção oposta de H2-H4.** Diferente de `le`/`test`/`tha`, aqui não tem sinal escondido atrás do token, é defeito de limpeza, 100% de correspondência com a causa técnica, sem exceções pra investigar.

**Correção aplicada em `src/preprocessing.py`:** `clean_text` agora chama `html.unescape()` antes do resto da limpeza, decodificando `&amp;` → `&` (e qualquer outra entidade HTML padrão) antes do texto ser reduzido a letras e espaços. Teste de regressão adicionado em `tests/test_preprocessing.py`. Como o pipeline já roda corrigido desde a seção 4 deste notebook.

Efeito da correção (-92 tokens, -1 vocabulário: 4.189→4.188).

## 6. Reviews sem tokens após o pré-processamento

In [96]:
empty_mask = df["n_tokens_proc"] == 0
print(f"Reviews sem tokens após pré-processamento: {empty_mask.sum()}")
print()
print(df[empty_mask][["review", "sentiment"]].to_string())


Reviews sem tokens após pré-processamento: 14

                                                                                                                                                                                                              review sentiment
129                                                                                                                                                                                                  जादा ठिक नही था  positivo
130                                                                                                                                                                                                     अच्छा नही था  positivo
135                                                                                                                                            🙏🏻જય સ્વામિનારાયણ<br/><br/>બહુ સરસ કોલ્ડ કોકો ક્વોલિટી બહુ સારી આભાર     neutro
457                                                          

Textos em Hindi/Gujarati puro ou compostos só por stopwords (`"same as above"`, `"it wasn't as before"`). Sem tokens, sem sinal para o TF-IDF, removidos na etapa de salvamento.


## 7. Salvar dataset processado

In [97]:
cols_to_save = ["rating", "review", "sentiment", "review_processed"]
output_path = "../data/processed/zomato_reviews_processed.csv"

_PANDAS_NA = {"nan", "NaN", "NA", "N/A", "n/a", "null", "NULL", "", "#N/A", "#NA"}

n_before = len(df)
n_empty_tokens = (df["n_tokens_proc"] == 0).sum()
n_nan_string = df[df["n_tokens_proc"] > 0]["review_processed"].isin(_PANDAS_NA).sum()
print(f"Tokens vazios:       {n_empty_tokens}")
print(f"String nan residual: {n_nan_string}")

df_final = df[
    (df["n_tokens_proc"] > 0) &
    (~df["review_processed"].isin(_PANDAS_NA))
].copy()
n_dropped = n_before - len(df_final)
print(f"Total removidas: {n_dropped} reviews ({n_dropped/n_before*100:.1f}%)")
print(f"Shape salvo: {df_final[cols_to_save].shape}")
print(df_final["sentiment"].value_counts())

df_final[cols_to_save].to_csv(output_path, index=False, quoting=csv.QUOTE_NONNUMERIC)
print(f"\nSalvo: {output_path}")


Tokens vazios:       14
String nan residual: 1
Total removidas: 15 reviews (0.4%)
Shape salvo: (3355, 4)
sentiment
positivo    1636
negativo    1434
neutro       285
Name: count, dtype: int64

Salvo: ../data/processed/zomato_reviews_processed.csv


## 8. Conclusões e decisões para o nb03

### Achados

| Item | Valor |
|---|---|
| Shape de entrada | 3.370 reviews |
| Redução média de tokens por review | 12,9 → 7,7 (40,2% menos) |
| Vocabulário único | 4.188 tokens |
| Reviews sem tokens após o pipeline | 14 (Hindi/Gujarati puro ou só stopwords) |
| String `'nan'` residual | 1 |
| **Shape final salvo** | **3.355 reviews (99,6% do dataset de entrada)** |
| Distribuição final | positivo 1.636 / negativo 1.434 / neutro 285 |

### Decisões tomadas

- Pipeline reaproveitado de `src/preprocessing.py` (`clean_text`, `tokenize`, `remove_stopwords`, `lemmatize`)
- Lematização aceita mesmo sem POS tags, trade-off documentado desde a auditoria do nb01
- H1 confirma que a limpeza não distorce nenhuma classe desproporcionalmente, a decisão de aplicar o mesmo pipeline às três classes está respaldada por teste estatístico
- H2-H4 confirmam que `le`, `test` e `tha`, três tokens de alta frequência com forma estranha no top-20/ranking curto por classe, não são ruído do pipeline: `le` é artefato determinístico da lematização de "less" (96,8% das reviews com `le` contêm "less" no original), `test` é grafia Hinglish de "taste" (99,4% de exclusão mútua com o token "taste" na mesma review), `tha` é palavra funcional hindi (88 reviews, sempre em contexto Hinglish). Nenhum exige stopwords customizado nesta fase
- H5, ao contrário de H2-H4, confirma que `amp` **é** ruído do pipeline: resíduo da entidade HTML `&amp;` (68 reviews no texto bruto) não decodificada por `clean_text` antes da remoção de caracteres não alfabéticos. Corrigido com `html.unescape()` em `src/preprocessing.py`, removeu 92 ocorrências do token

### Próximo passo, notebook 03

- Vetorização TF-IDF (Format Data)
- Seleção explícita de features além de `min_df` (Select Data, nível coluna)
- Split de validação separado do teste na comparação de configurações/modelos